# 2 - Selección de variables

Qué materias primas explican el precio de cada equipo y cuáles son ruido.

In [1]:
import sys
sys.path.append("..")
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss

from src import datos

semanal = pd.read_csv(datos.PROCESSED / "equipos_semanal.csv", index_col="Date", parse_dates=True)
mensual = pd.read_csv(datos.PROCESSED / "equipos_mensual.csv", index_col="Date", parse_dates=True)
semanal.shape, mensual.shape

((713, 5), (164, 5))

## Estacionariedad

- **ADF**: H0 = tiene raíz unitaria (no estacionaria)
- **KPSS**: H0 = es estacionaria

Pruebo los precios en logaritmo y sus cambios (retornos logarítmicos).
Si las series tienen tendencia, correlacionarlas en niveles puede dar relaciones que no existen (regresión espuria). Uso dos pruebas que se complementan:

In [2]:
def estacionaria(s):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")  # avisos de statsmodels sobre la tabla de p-values
        p_adf = adfuller(s, autolag="AIC")[1]
        p_kpss = kpss(s, regression="c", nlags="auto")[1]
    if p_adf < 0.05 and p_kpss > 0.05:
        return "sí"
    if p_adf > 0.05 and p_kpss < 0.05:
        return "no"
    return "dudoso"

filas = []
for nombre, df in [("semanal", semanal), ("mensual", mensual)]:
    logp = np.log(df)
    ret = logp.diff().dropna()
    for col in df.columns:
        filas.append([nombre, col, estacionaria(logp[col]), estacionaria(ret[col])])

pd.DataFrame(filas, columns=["frecuencia", "serie", "niveles", "retornos"]).pivot(
    index="serie", columns="frecuencia", values=["niveles", "retornos"])

niveles         retornos        
frecuencia mensual semanal  mensual semanal
serie                                      
Equipo1     dudoso      no       sí      sí
Equipo2     dudoso      no       sí      sí
X               no      no       sí      sí
Y           dudoso      no       sí      sí
Z           dudoso      no       sí      sí

En niveles ninguna es claramente estacionaria y en retornos todas lo son. Las relaciones de corto plazo las mido con retornos.

## Niveles vs retornos

In [3]:
ret_m = np.log(mensual).diff().dropna()

comparacion = pd.concat({
    "niveles": mensual.corr().loc[["Equipo1", "Equipo2"], ["X", "Y", "Z"]],
    "retornos": ret_m.corr().loc[["Equipo1", "Equipo2"], ["X", "Y", "Z"]],
}, axis=1)
comparacion.round(2)

niveles             retornos            
              X     Y     Z        X     Y     Z
Equipo1    0.53  1.00  0.85     0.33  1.00  0.44
Equipo2    0.53  0.92  0.99     0.50  0.65  0.96

En niveles parece que todo se relaciona con todo (Equipo1 vs Z = 0.85, Equipo2 vs Y = 0.92). En retornos esas correlaciones bajan bastante, porque buena parte venía de que todos los commodities suben y bajan juntos en el ciclo.

## Regresión en retornos

Regresión de los retornos de cada equipo contra los retornos de X, Y y Z al mismo tiempo. Uso errores estándar HAC (Newey-West) porque en series de tiempo los residuos suelen estar autocorrelacionados.

In [4]:
def regresion(df, equipo, lags_hac):
    r = np.log(df).diff().dropna()
    m = sm.OLS(r[equipo], sm.add_constant(r[["X", "Y", "Z"]])).fit(
        cov_type="HAC", cov_kwds={"maxlags": lags_hac})
    return m

resultados = {}
for freq, df, lags in [("semanal", semanal, 4), ("mensual", mensual, 3)]:
    for eq in ["Equipo1", "Equipo2"]:
        m = regresion(df, eq, lags)
        resultados[(freq, eq)] = {
            "coef X": m.params["X"], "coef Y": m.params["Y"], "coef Z": m.params["Z"],
            "p X": m.pvalues["X"], "p Y": m.pvalues["Y"], "p Z": m.pvalues["Z"],
            "R2": m.rsquared,
        }

pd.DataFrame(resultados).T.round(3)

coef X  coef Y  coef Z   p X  p Y    p Z     R2
semanal Equipo1   0.037   0.989   0.018  0.00  0.0  0.270  0.781
        Equipo2   0.021   0.199   0.764  0.01  0.0  0.000  0.769
mensual Equipo1   0.024   0.966   0.002  0.00  0.0  0.856  0.993
        Equipo2   0.031   0.201   0.763  0.00  0.0  0.000  0.989

In [5]:
# detalle del modelo mensual de cada equipo
print(regresion(mensual, "Equipo1", 3).summary().tables[1])
print(regresion(mensual, "Equipo2", 3).summary().tables[1])

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const       9.469e-06      0.000      0.040      0.968      -0.000       0.000
X              0.0245      0.005      4.504      0.000       0.014       0.035
Y              0.9662      0.006    154.416      0.000       0.954       0.978
Z              0.0019      0.010      0.181      0.856      -0.019       0.022
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const       6.278e-05      0.000      0.327      0.744      -0.000       0.000
X              0.0306      0.004      7.099      0.000       0.022       0.039
Y              0.2012      0.009     21.735      0.000       0.183       0.219
Z              0.7631      0.009     83.227      0.000       0.745       0.781


**Equipo1** depende casi solo de Y (coeficiente ~0.97). Z no es significativo (p = 0.86), aunque en niveles tenía correlación de 0.85.

**Equipo2** depende de Z (~0.76) y en menor medida de Y (~0.20).

**X** sale significativo en los dos pero con coeficientes de 0.02-0.03, o sea, un cambio de 10% en X mueve al equipo apenas 0.2-0.3%. Lo dejo pendiente y lo reviso más abajo con la prueba de cointegración.

En ambos equipos los coeficientes suman casi 1. Eso es lo que esperaría si el precio del equipo fuera un índice ponderado de sus insumos: si todos suben 10%, el equipo sube 10%.

## Semanal o mensual

In [6]:
pd.DataFrame(resultados).T["R2"].unstack().round(3)

,Equipo1,Equipo2
mensual,0.993,0.989
semanal,0.781,0.769


Con datos semanales el R² es ~0.78 y con mensuales ~0.99. El precio diario de los equipos trae ruido propio que no viene de las materias primas, y al promediar por mes ese ruido se cancela.

Me quedo con **mensual**: explica mucho mejor y además es la frecuencia que sirve para la planeación financiera del proyecto.

## Rezagos

Si un equipo reaccionara a una materia prima uno o dos meses después, podría pronosticar esos meses con datos que ya conozco. Reviso la correlación de los retornos del equipo en el mes t contra los de cada insumo en t-k.

In [7]:
filas = {}
for eq in ["Equipo1", "Equipo2"]:
    for ins in ["X", "Y", "Z"]:
        filas[(eq, ins)] = {k: ret_m[eq].corr(ret_m[ins].shift(k)) for k in range(-2, 5)}

pd.DataFrame(filas).T.round(2)

-2    -1     0     1     2     3     4
Equipo1 X  0.09  0.32  0.33  0.28  0.06 -0.13 -0.13
        Y -0.05  0.42  1.00  0.43 -0.05 -0.06  0.01
        Z -0.01  0.17  0.44  0.40  0.15 -0.05  0.05
Equipo2 X  0.08  0.25  0.50  0.25  0.05 -0.11 -0.13
        Y  0.10  0.47  0.65  0.27 -0.03 -0.04  0.04
        Z  0.00  0.25  0.96  0.32  0.03 -0.01  0.01

La correlación fuerte está en k=0. Los valores de k=1 y k=-1 son parecidos entre sí, lo que apunta a un efecto del promedio mensual (dos meses consecutivos comparten parte de la información) más que a una reacción retrasada. Lo confirmo metiendo los rezagos en la regresión:

In [8]:
def regresion_rezagos(equipo, n_rezagos=3):
    X = pd.concat({f"{ins}_t-{k}": ret_m[ins].shift(k)
                   for ins in ["X", "Y", "Z"] for k in range(n_rezagos + 1)}, axis=1).dropna()
    m = sm.OLS(ret_m.loc[X.index, equipo], sm.add_constant(X)).fit(
        cov_type="HAC", cov_kwds={"maxlags": 3})
    return m.pvalues.drop("const")

pvals = pd.DataFrame({eq: regresion_rezagos(eq) for eq in ["Equipo1", "Equipo2"]})
pvals.round(3)

,Equipo1,Equipo2
X_t-0,0.001,0.000
X_t-1,0.849,0.495
X_t-2,0.835,0.750
X_t-3,0.822,0.389
Y_t-0,0.000,0.000
Y_t-1,0.513,0.113
Y_t-2,0.801,0.071
Y_t-3,0.469,0.551
Z_t-0,0.759,0.000
Z_t-1,0.585,0.578


Ningún rezago es significativo: los equipos reaccionan a sus insumos **en el mismo mes**. Esto tiene una consecuencia práctica, no hay meses "gratis" de pronóstico: para proyectar el costo de los equipos primero hay que proyectar las materias primas.

## Cointegración

Las series no son estacionarias en niveles, pero si existe una combinación de ellas que sí lo sea, hay una relación de equilibrio de largo plazo (cointegración). Uso la prueba de Engle-Granger con y sin X.

In [9]:
from statsmodels.tsa.stattools import coint

log_m = np.log(mensual)
pruebas = [
    ("Equipo1", ["Y"]),
    ("Equipo1", ["X", "Y"]),
    ("Equipo2", ["Y", "Z"]),
    ("Equipo2", ["X", "Y", "Z"]),
]

pd.DataFrame(
    [[eq, " + ".join(ins), coint(log_m[eq], log_m[ins])[1]] for eq, ins in pruebas],
    columns=["equipo", "insumos", "p-value"],
).round(4)

,equipo,insumos,p-value
0,Equipo1,Y,0.2073
1,Equipo1,X + Y,0.0000
2,Equipo2,Y + Z,0.3507
3,Equipo2,X + Y + Z,0.0009


Sin X **no** hay cointegración (p = 0.21 y 0.35). Con X sí, y muy clara (p < 0.001).

Esto resuelve lo que dejé pendiente: X mueve poco a los equipos en términos de porcentaje, pero sin X la relación de largo plazo no cierra. **X se queda.**

## La fórmula del costo

Como la relación es contemporánea y de equilibrio, pruebo una regresión directa en niveles (precios, sin logaritmos).

In [10]:
formulas = {}
for eq in ["Equipo1", "Equipo2"]:
    m = sm.OLS(mensual[eq], sm.add_constant(mensual[["X", "Y", "Z"]])).fit(
        cov_type="HAC", cov_kwds={"maxlags": 3})
    ic = m.conf_int()
    formulas[eq] = pd.DataFrame({"coef": m.params, "ic 95% inf": ic[0], "ic 95% sup": ic[1], "p": m.pvalues})
    print(eq, "| R2:", round(m.rsquared, 4), "| desv. residuos:", round(m.resid.std(), 2))
    print("   const = 0 y X + Y + Z = 1  ->  p =", round(float(m.f_test("const = 0, X + Y + Z = 1").pvalue), 3))

pd.concat(formulas).round(4)

Equipo1 | R2: 0.9997 | desv. residuos: 1.81
   const = 0 y X + Y + Z = 1  ->  p = 0.984
Equipo2 | R2: 0.9997 | desv. residuos: 2.68
   const = 0 y X + Y + Z = 1  ->  p = 0.769


coef  ic 95% inf  ic 95% sup       p
Equipo1 const  0.0328     -1.3896      1.4551  0.9640
        X      0.1983      0.1821      0.2145  0.0000
        Y      0.8006      0.7951      0.8060  0.0000
        Z     -0.0001     -0.0018      0.0016  0.9045
Equipo2 const  0.6720     -1.4174      2.7614  0.5284
        X      0.3358      0.3175      0.3540  0.0000
        Y      0.3353      0.3298      0.3408  0.0000
        Z      0.3322      0.3301      0.3342  0.0000

Los coeficientes son prácticamente fracciones exactas:

- **Equipo1 = 0.2·X + 0.8·Y**
- **Equipo2 = ⅓·X + ⅓·Y + ⅓·Z**

La constante no es significativa, Z no aparece en Equipo1, y la prueba conjunta no rechaza que la constante sea 0 y que los pesos sumen 1. El precio de cada equipo se comporta como una "receta" de insumos: Equipo1 lleva 20% de X y 80% de Y; Equipo2 lleva partes iguales de los tres.

Esto también explica la regresión en retornos de arriba: X tiene un peso de 20% y 33% en la fórmula, pero como su precio (~78) es mucho menor que el de Y (~555) o Z (~2,040), su efecto en porcentaje sale chico (elasticidad ≈ peso × precio del insumo / precio del equipo).

## Resumen

| | Insumos relevantes | Fórmula | Rezago |
|---|---|---|---|
| Equipo1 | X, Y | 0.2·X + 0.8·Y | ninguno |
| Equipo2 | X, Y, Z | (X + Y + Z) / 3 | ninguno |

- Z es ruido para Equipo1, aunque en niveles su correlación era 0.85.
- Trabajo en frecuencia mensual.
- Para proyectar los equipos necesito proyectar X, Y y Z. En el notebook 3 elijo el modelo y en el 4 hago el pronóstico.